Randomised Riemannian Hamiltonian Monte Carlo for Sampling on $\mathbb{V}_{d,p}$

Defined by $g(X) = X^{T}X - \mathbb{I}_{p} = 0$, we have $\mathcal{M} := \{x \in \mathbb{R}^{d \times p} \mid g(x) = 0\}$

$T_{X}\mathcal{M} =  \{V \in \mathbb{R}^{d \times p} \mid V^{T}X + X^{T}V = 0 \}$

For $A \in \mathbb{R}^{d \times p}$ with $v_{i} \in \mathbb{R}^{d}$, $i = 1,...,p$ being the column vectors. We define $g_{ij}(A) = v_{i} \cdot v_{j} - \delta_{ij} = 0$


1) Dynamics

We are sampling from the von Mises-Fisher distribution which has density given by 

$p_{vMF}(X) \propto \exp{(\langle f_{1},x_{1} \rangle + ... + \langle f_{p},x_{p}\rangle)}$

In [ ]:
import numba
import numpy as np
from numpy.linalg import inv
import matplotlib.pyplot as plt
import matplotlib.animation as animation
import time
import random

#p<=d
p=3
d=3
dimension = int(d*p)
num_of_constraints  = int(p*(p+1)/2)

#Matrix in distribution

F= np.array([[0,2,-45],[-2,0,-4],[45,4,0]])

@numba.jit(nopython=True)
def vec_to_matrix(q):
    X = np.zeros((d,p))
    for i in range(d):
        for j in range(p):
            X[i,j] = q[j*d+i]
    return X

@numba.jit(nopython=True)
def matrix_to_vec(X):
    #initialising filler array
    x = np.zeros(d*p)
    
    for i in range(d*p):
        i_index = i%d
        j_index =  int((i - i_index)/d)
        x[i] = X[i_index,j_index]
    return x

@numba.jit(nopython=True)
def dot_product(v1,v2):
    dot = 0
    for i in range(len(v1)):
        
        dot += v1[i]*v2[i]
        
    return dot

@numba.jit(nopython=True)
def matmul(matrix1,matrix2):
    a = matrix1.shape[0]
    b = matrix2.shape[1]
    c = matrix2.shape[0]
    rmatrix = np.zeros((a,b))
    for i in range(a):
        for j in range(b):
            for k in range(c):
                rmatrix[i,j] += matrix1[i,k] * matrix2[k,j]
    return rmatrix

@numba.jit(nopython=True)
def matrix_vec_multiplication(A,x):
    v = np.zeros(len(A))
    
    for i in range(len(A)):
        for j in range(len(x)):
                v[i] += A[i][j] * x[j]
    return v


@numba.jit(nopython=True)
def g_ij(q,i,j):
    
    X = vec_to_matrix(q)
    
    if i==j:
        y = np.linalg.norm(X[:,i])**2 - 1
    else:
        y = dot_product(X[:,i],X[:,j])
        
    return y

@numba.jit(nopython=True)
def G_matrix_input(X): #considering i<j.
    
    
    z = np.zeros((dimension,num_of_constraints))
    
    for i in range(p): #block diagonals
        z[d*i:d*(i+1),int(p*i-0.5*i*(i-1)):int(p*(i+1) - 0.5*i*(i+1))] = X[:,i:]
    
        #vector diagonals
        for j in range(p-i):
            z[(j+i)*d:(j+i+1)*d,int(p*i-0.5*i*(i-1) + j)] += X[:,i]
    z = z.T    
    return z

@numba.jit(nopython=True)
def G(q): #considering i<j.
    
    
    
    X = vec_to_matrix(q)
    
    z = np.zeros((dimension,num_of_constraints))
    
    for i in range(p): #block diagonals
        z[d*i:d*(i+1),int(p*i-0.5*i*(i-1)):int(p*(i+1) - 0.5*i*(i+1))] = X[:,i:]
    
        #vector diagonals
        for j in range(p-i):
            z[(j+i)*d:(j+i+1)*d,int(p*i-0.5*i*(i-1) + j)] += X[:,i]  
    z = z.T #could implement this above
    return z

Fvec = matrix_to_vec(F)

@numba.jit(nopython=True)
def potential_derv(q):
    mu = -Fvec
    return mu

#RATTLE Hamiltonian Flow
@numba.jit(nopython=True)
def RATTLE_with_Potential(x0,v0,t,dt,max_elim_iters):
    n = np.floor(t/dt)
    vn = v0
    qn = x0
    vhalf = v0
    G_q = G(qn)
    
    #Gram Matrix is GG^T
    gram = matmul(G_q,G_q.T)
    gram_inv = np.linalg.inv(gram)
    
    pderv = potential_derv(qn)
    
    residual_list = np.zeros(num_of_constraints)
    for i in range(int(n)):
        
        
        #solver for Lagrange position multipliers
        Q = qn + vn*dt - 0.5*dt*dt*pderv
        
        #non-linear gaussian elimination
        for k in range(max_elim_iters): #i>j
            for i in range(p):
                for j in range(i,p):
                    g_Q = g_ij(Q,i,j)
                    index = int(i*p - 0.5*i*(i-1) + j-i)
                    
                    residual_list[index] = g_Q
                    if abs(g_Q) < 1e-8:
                        continue
                    G_Q = G(Q)
                    
                    #should be sum of i's and js in indexing below
                    dlambda = g_Q/dot_product(G_Q[index,:],G_q[index,:])
                    Q = Q - G_q[index,:]*dlambda
            #break condition
            if np.all(np.abs(residual_list)<1e-8):
                break

        
        #half step
        vhalf = (Q-qn)/dt
        qn = Q
        
        pderv =  potential_derv(qn)
        G_q = G(qn)
        
        gram = matmul(G_q,G_q.T)
        gram_inv = np.linalg.inv(gram)
        
        #linear solver Lagrange velocity multipliers
        b = matrix_vec_multiplication(G_q,2*vhalf/dt - pderv)
        coeffs_v = matrix_vec_multiplication(gram_inv,b)
        
        #full step
        vn = vhalf - 0.5*dt*pderv - 0.5*dt*matrix_vec_multiplication(G_q.T,coeffs_v)
       
    return qn,vn




#RATTLE Hamiltonian Flow
@numba.jit(nopython=True)
def Geodesic(x0,v0,n,dt,max_elim_iters):
    vn = v0
    qn = x0
    vhalf = v0
    G_q = G(qn)
    
    #Gram Matrix is GG^T
    gram = matmul(G_q,G_q.T)
    gram_inv = np.linalg.inv(gram)

    
    residual_list = np.zeros(num_of_constraints)
    for i in range(int(n)):
        
        
        #solver for Lagrange position multipliers
        Q = qn + vn*dt
        
        #non-linear gaussian elimination
        for k in range(max_elim_iters): #i>j
            for i in range(p):
                for j in range(i,p):
                    g_Q = g_ij(Q,i,j)
                    index = int(i*p - 0.5*i*(i-1) + j-i)
                    
                    residual_list[index] = g_Q
                    if abs(g_Q) < 1e-8:
                        continue
                    G_Q = G(Q)
                    
                    #should be sum of i's and js in indexing below
                    dlambda = g_Q/dot_product(G_Q[index,:],G_q[index,:])
                    Q = Q - G_q[index,:]*dlambda
            #break condition
            if np.all(np.abs(residual_list)<1e-8):
                break

        
        #half step
        vhalf = (Q-qn)/dt
        qn = Q
        
        G_q = G(qn)
        
        gram = matmul(G_q,G_q.T)
        gram_inv = np.linalg.inv(gram)
        
        #linear solver Lagrange velocity multipliers
        b = matrix_vec_multiplication(G_q,2*vhalf/dt)
        coeffs_v = matrix_vec_multiplication(gram_inv,b)
        
        #full step
        vn = vhalf - 0.5*dt*matrix_vec_multiplication(G_q.T,coeffs_v)
       
    return qn,vn

In [ ]:
@numba.jit(nopython=True)
def tangent_space_gaussian(q):
    
    
    v = np.random.normal(0,1,dimension).T
    
    G_q = G(q)
    
    
    gram = matmul(G_q,G_q.T)
    gram_inv = np.linalg.inv(gram)
    
    proj_matrix = np.eye(dimension) - matmul(G_q.T,matmul(gram_inv,G_q))
    
    #sample 3d gaussian and then project onto tangent space.
    v = matrix_vec_multiplication(proj_matrix,v)
    
    return v

#check
X = np.eye(d,p)
X_vec = matrix_to_vec(X)
Z = tangent_space_gaussian(X_vec)
T_X = vec_to_matrix(Z)  

print(matmul(T_X.T,X) + matmul(X.T,T_X))

In [ ]:
#RATTLE Hamiltonian Flow
@numba.jit(nopython=True)
def g_BAOAB(γ,x0,v0,Kr,Kp,dt):
    
    pn = v0
    qn = x0
    
    G_q = G(qn)
    gram = matmul(G_q,G_q.T)
    gram_inv = np.linalg.inv(gram)
    
    a = np.exp(-γ*dt) 
    
    b = np.sqrt(1-a**2)
    
    pderv = potential_derv(qn)
    
    for i in range(Kp):
        
        
        #B
        μ = (dt/2)*matrix_vec_multiplication(G_q,pderv)
        μ = matrix_vec_multiplication(gram_inv,μ)
        
        pn = pn - dt*pderv/2 + matrix_vec_multiplication(G_q.T,μ)
        
        #A
        [qn,pn] = Geodesic(qn,pn,Kr,dt/(2*Kr),50)
        
        
        #O
        G_q = G(qn)
        gram = matmul(G_q,G_q.T)
        gram_inv = np.linalg.inv(gram)
        
        gaussian = tangent_space_gaussian(qn)
        
        
        
        μ = -b*matrix_vec_multiplication(G_q,gaussian)
        μ = matrix_vec_multiplication(gram_inv,μ)
        
        pn = a*pn + b*gaussian + matrix_vec_multiplication(G_q.T,μ)
        
        
        #A
        [qn,pn] = Geodesic(qn,pn,Kr,dt/(2*Kr),50)
        
        
        #B
        G_q = G(qn)
        gram = matmul(G_q,G_q.T)
        gram_inv = np.linalg.inv(gram)
        
        pderv = potential_derv(qn)
        
        μ = (dt/2)*matrix_vec_multiplication(G_q,pderv)
        μ = matrix_vec_multiplication(gram_inv,μ)
        
        pn = pn - dt*pderv/2 + matrix_vec_multiplication(G_q.T,μ)
            
    return qn,pn

In [ ]:
print(F)

In [ ]:
# x = matrix_to_vec(np.eye(d, p))
# v = tangent_space_gaussian(x)
# print(g_BAOAB(2.,x,v,1,1,0.01))

# Checks

In [ ]:
X = np.array([[1,2,3], [4,5,6],[7,8,9]])
#print(X[:,1:])
#g_ij checked
#G checked for what I want.
#print(X.T,X)
M = np.eye(d, p)
M = matrix_to_vec(M)
gram = matmul(G(M),G(M).T)
"G matrix should be #constraints by dimension 0.5*p*(p+1) by p*d" 
#6 by 9
print(gram)
M = np.eye(d, p)
print(matmul(G_matrix_input(M),G_matrix_input(M).T))


2) Event Time Sampling


In [ ]:
#Sampling Event Times
@numba.jit(nopython=True)
def time_exp(lam):
    t = np.random.exponential(lam)
    return t

In [ ]:
@numba.jit(nopython=True)
def f(q):
    
    z = -dot_product(Fvec,q)
    
    return z

@numba.jit(nopython=True)
def hamiltonian(x,v):
    return f(x) + 0.5*dot_product(v,v)

4) Simulation

In [ ]:
@numba.jit(nopython=True)
def RRHMC(num_of_events,dt_max,T,x_init):
    
    #Exponential Expected Value
    rate = T
    
    #initialisation of x on V_{d,p}
    x = x_init
    v = tangent_space_gaussian(x)
    
    position_list = [x]
    gradient_evaluations = 0
    
    for i in range(num_of_events):
        
        t = time_exp(rate)
        L = np.ceil(t/dt_max)
        
        gradient_evaluations += L
        dt = t/L
        
        h = hamiltonian(x,v)
    
        xnew,vnew = RATTLE_with_Potential(x,v,t,dt,50)
        
        h_new = hamiltonian(xnew,vnew)
        
        #metropolis hasting step
        u = np.random.rand()
        if u <= np.exp(-h_new+h):
            x = xnew
            

        
        position_list.append(x)
        
        v = tangent_space_gaussian(x)
        if i%10000==0:
            print(i)
    gradient_evaluations /= num_of_events
        
        
    return position_list, gradient_evaluations

In [ ]:
@numba.jit(nopython=True)
def RHMC(num_of_events,dt,T,x_init):
    
    #initialisation of x on V_{d,p}
    x = x_init
    v = tangent_space_gaussian(x)
    
    position_list = [x]
    
    
    for i in range(num_of_events):
        
        h = hamiltonian(x,v)
        
        xnew,vnew = RATTLE_with_Potential(x,v,T,dt,50)
        
        h_new = hamiltonian(xnew,vnew)
        
        #metropolis hasting step
        u = np.random.rand()
        if u <= np.exp(-h_new+h):
            x = xnew
        
        position_list.append(x)
        v = tangent_space_gaussian(x)  
        if i%10000==0:
            print(i)
            
    return position_list

In [ ]:
@numba.jit(nopython=True)
def g_BAOAB_MC(γ,Kr,num_of_events,dt,x_init):
   
    
    #initialisation of x on V_{d,p}
    x = x_init
    v = tangent_space_gaussian(x)
    
    position_list = [x]
    N = int(t/dt)
    
    for i in range(num_of_events):
        x,v = g_BAOAB(γ,x,v,Kr,1,dt)
        
        position_list.append(x)
        
        if i%10000==0:
            print(i)
    return position_list

In [ ]:
#Initialise
x_init = matrix_to_vec(np.eye(d,p))
num_of_events = 1000
dt = 0.01

t = time.time()
position, gradient_evaluations = RRHMC(num_of_events,dt,2*dt,x_init)
elapsed = time.time() - t
print('RT time =',elapsed)
t = time.time()
position = RHMC(num_of_events,dt,2*dt,x_init)
elapsed = time.time() - t
print('DT time =',elapsed)
t = time.time()
position = g_BAOAB_MC(2.,1,num_of_events,dt,x_init)
elapsed = time.time() - t
print('g-BAOAB time =',elapsed)


position_matrix = []
for i in position:
    position_matrix.append(vec_to_matrix(i))

# Check

In [ ]:
constraint_list = []

for i in position_matrix:
    constraint_list.append(np.linalg.norm(np.matmul(i.T,i)-np.eye(p)))
plt.plot(constraint_list)
print('Maximum Constraint Error =',max(constraint_list))

In [ ]:
Num = 100
N_h = 5
dt_ = 0.16


  
mean_list_RT = []
mean_list_DT = []
mean_list_g_BAOAB = []
    
T = N_h*dt_
    
two_DT_samples = RHMC(Num,dt_,T,x_init)
two_RT_samples, gradient_evaluations = RRHMC(Num,dt_,T,x_init)
g_BAOAB_samples = g_BAOAB_MC(2.,1,Num,0.2,x_init)

    
two_DT_means = []
two_RT_means = []
g_BAOAB_means = []
    
for i in two_DT_samples:

    mean_list_DT.append(f(i))
    two_DT_means.append(np.mean(mean_list_DT))

mean_estimate_DT = two_DT_means[-1]

print('DT Mean Estimate =', mean_estimate_DT)

for i in two_RT_samples:

    mean_list_RT.append(f(i))
    two_RT_means.append(np.mean(mean_list_RT))

mean_estimate_RT = two_RT_means[-1]

print('RT Mean Estimate =', mean_estimate_RT)

for i in g_BAOAB_samples:

    mean_list_g_BAOAB.append(f(i))
    g_BAOAB_means.append(np.mean(mean_list_g_BAOAB))

mean_estimate_g_BAOAB = g_BAOAB_means[-1]

print('g-BAOAB Mean Estimate =', mean_estimate_g_BAOAB)



In [ ]:
burn = 10
#plt.plot(two_DT_means[burn:],label = 'DT-mean')
plt.plot(two_RT_means[burn:],label = 'RT-mean')
plt.plot(g_BAOAB_means[burn:],label = 'g-BAOAB-mean')

plt.legend()
plt.show()

In [ ]:
import statsmodels.api as sm
#ESS per gradient evaluation

N_samples = 10000


dt_list = np.linspace(0.00001,0.05,100)
print(dt_list)

#Gradient_p_ESS_DT_1 = []
Gradient_p_ESS_RT_1 = []

#Gradient_p_ESS_DT_2 = []
Gradient_p_ESS_RT_2 = []

#Gradient_p_ESS_DT_5 = []
Gradient_p_ESS_RT_5 = []

#Gradient_p_ESS_DT_10 = []
Gradient_p_ESS_RT_10 = []

Gradient_p_ESS_g_BAOAB_2 = []
Gradient_p_ESS_g_BAOAB_0_5 = []
Gradient_p_ESS_g_BAOAB_10 = []


for dt_ in dt_list:
    
    T = 5*dt_
    
    #DT_samples_5 = RHMC(N_samples,dt_,T,x_init)
    RT_samples_5, gradient_evaluations_5 = RRHMC(N_samples,dt_,T,x_init)
    
    g_BAOAB_samples_2 = g_BAOAB_MC(2.,1,N_samples,dt_,x_init)
    g_BAOAB_samples_0_5 = g_BAOAB_MC(0.5,1,N_samples,dt_,x_init)
    g_BAOAB_samples_10 = g_BAOAB_MC(10.,1,N_samples,dt_,x_init)
    
    T = 1*dt_
    
    #DT_samples_1 = RHMC(N_samples,dt_,T,x_init)
    RT_samples_1, gradient_evaluations_1 = RRHMC(N_samples,dt_,T,x_init)
    
    
    T = 2*dt_
    
    #DT_samples_2 = RHMC(N_samples,dt_,T,x_init)
    RT_samples_2, gradient_evaluations_2 = RRHMC(N_samples,dt_,T,x_init)
    
    
    
    T = 10*dt_
    
    #DT_samples_10 = RHMC(N_samples,dt_,T,x_init)
    RT_samples_10, gradient_evaluations_10 = RRHMC(N_samples,dt_,T,x_init)
    
    #####################################################################
#     #Gradient Evaluations per ESS - DT - 1
#     front = len(DT_samples_1)//10
    
#     N = len(DT_samples_1)- front

#     auto_corr_vector_DT_1 = []
    
#     for i in DT_samples_1:
        
#         auto_corr_vector_DT_1.append(f(i))
    
#     auto_DT_1 = sm.tsa.acf(auto_corr_vector_DT_1[front:],nlags = N//50)
    
#     IAC_DT_1 = np.sum(auto_DT_1)
    
#     print('IAC_DT=',2*IAC_DT_1-1)
    
#     ESS_percent_DT_1 = 1/(2*IAC_DT_1 - 1)
    
#     Gradient_p_ESS_DT_1.append(1./ESS_percent_DT_1)
    
    #Gradient Evaluations per ESS - RT - 1
    front = len(RT_samples_1)//10
    
    N = len(RT_samples_1)- front

    auto_corr_vector_RT_1 = []
    
    for i in RT_samples_1:
        
        auto_corr_vector_RT_1.append(f(i))
    
    auto_RT_1 = sm.tsa.acf(auto_corr_vector_RT_1[front:],nlags = N//50)
    
    IAC_RT_1 = np.sum(auto_RT_1)
    
    print('IAC_RT=',2*IAC_RT_1-1)
    
    ESS_percent_RT_1 = 1/(2*IAC_RT_1 - 1)
    
    Gradient_p_ESS_RT_1.append(gradient_evaluations_1/ESS_percent_RT_1)
    
    
    
    
    #####################################################################
#     #Gradient Evaluations per ESS - DT - 2
#     front = len(DT_samples_2)//10
    
#     N = len(DT_samples_2)- front

#     auto_corr_vector_DT_2 = []
    
#     for i in DT_samples_2:
        
#         auto_corr_vector_DT_2.append(f(i))
    
#     auto_DT_2 = sm.tsa.acf(auto_corr_vector_DT_2[front:],nlags = N//50)
    
#     IAC_DT_2 = np.sum(auto_DT_2)
    
#     print('IAC_DT=',2*IAC_DT_2-1)
    
#     ESS_percent_DT_2 = 1/(2*IAC_DT_2 - 1)
    
#     Gradient_p_ESS_DT_2.append(2./ESS_percent_DT_2)
    
    #Gradient Evaluations per ESS - RT - 2
    front = len(RT_samples_2)//10
    
    N = len(RT_samples_2)- front

    auto_corr_vector_RT_2 = []
    
    for i in RT_samples_2:
        
        auto_corr_vector_RT_2.append(f(i))
    
    auto_RT_2 = sm.tsa.acf(auto_corr_vector_RT_2[front:],nlags = N//50)
    
    IAC_RT_2 = np.sum(auto_RT_2)
    
    print('IAC_RT=',2*IAC_RT_2-1)
    
    ESS_percent_RT_2 = 1/(2*IAC_RT_2 - 1)
    
    Gradient_p_ESS_RT_2.append(gradient_evaluations_2/ESS_percent_RT_2)
    
    
    
    
    #####################################################################
#     #Gradient Evaluations per ESS - DT - 5
#     front = len(DT_samples_5)//10
    
#     N = len(DT_samples_5)- front

#     auto_corr_vector_DT_5 = []
    
#     for i in DT_samples_5:
        
#         auto_corr_vector_DT_5.append(f(i))
    
#     auto_DT_5 = sm.tsa.acf(auto_corr_vector_DT_5[front:],nlags = N//50)
    
#     IAC_DT_5 = np.sum(auto_DT_5)
    
#     print('IAC_DT=',2*IAC_DT_5-1)
    
#     ESS_percent_DT_5 = 1/(2*IAC_DT_5 - 1)
    
#     Gradient_p_ESS_DT_5.append(5./ESS_percent_DT_5)
    
    #Gradient Evaluations per ESS - RT - 5
    front = len(RT_samples_5)//10
    
    N = len(RT_samples_5)- front

    auto_corr_vector_RT_5 = []
    
    for i in RT_samples_5:
        
        auto_corr_vector_RT_5.append(f(i))
    
    auto_RT_5 = sm.tsa.acf(auto_corr_vector_RT_5[front:],nlags = N//50)
    
    IAC_RT_5 = np.sum(auto_RT_5)
    
    print('IAC_RT=',2*IAC_RT_5-1)
    
    ESS_percent_RT_5 = 1/(2*IAC_RT_5 - 1)
    
    Gradient_p_ESS_RT_5.append(gradient_evaluations_5/ESS_percent_RT_5)
    
    
    
    #####################################################################
#     #Gradient Evaluations per ESS - DT - 10
#     front = len(DT_samples_10)//10
    
#     N = len(DT_samples_10)- front

#     auto_corr_vector_DT_10 = []
    
#     for i in DT_samples_10:
        
#         auto_corr_vector_DT_10.append(f(i))
    
#     auto_DT_10 = sm.tsa.acf(auto_corr_vector_DT_10[front:],nlags = N//50)
    
#     IAC_DT_10 = np.sum(auto_DT_10)
    
#     print('IAC_DT=',2*IAC_DT_10-1)
    
#     ESS_percent_DT_10 = 1/(2*IAC_DT_10 - 1)
    
#     Gradient_p_ESS_DT_10.append(10./ESS_percent_DT_10)
    
    #Gradient Evaluations per ESS - RT - 10
    front = len(RT_samples_10)//10
    
    N = len(RT_samples_10)- front

    auto_corr_vector_RT_10 = []
    
    for i in RT_samples_10:
        
        auto_corr_vector_RT_10.append(f(i))
    
    auto_RT_10 = sm.tsa.acf(auto_corr_vector_RT_10[front:],nlags = N//50)
    
    IAC_RT_10 = np.sum(auto_RT_10)
    
    print('IAC_RT=',2*IAC_RT_10-1)
    
    ESS_percent_RT_10 = 1/(2*IAC_RT_10 - 1)
    
    Gradient_p_ESS_RT_10.append(gradient_evaluations_10/ESS_percent_RT_10)
    
    
    
    ###################################################################
    
    #Gradient Evaluations per ESS - g-BAOAB - γ = 2
    front = len(g_BAOAB_samples_2)//10
    
    N = len(g_BAOAB_samples_2)- front

    auto_corr_vector_g_BAOAB_2 = []
    
    for i in g_BAOAB_samples_2:
        
        auto_corr_vector_g_BAOAB_2.append(f(i))
    
    auto_g_BAOAB_2 = sm.tsa.acf(auto_corr_vector_g_BAOAB_2[front:],nlags = N//50)
    
    IAC_g_BAOAB_2 = np.sum(auto_g_BAOAB_2)
    
    print('IAC_g_BAOAB=',2*IAC_g_BAOAB_2-1)
    
    ESS_percent_g_BAOAB_2 = 1/(2*IAC_g_BAOAB_2 - 1)
    
    Gradient_p_ESS_g_BAOAB_2.append(1/ESS_percent_g_BAOAB_2)
    
    
    #Gradient Evaluations per ESS - g-BAOAB - γ = 0.5
    front = len(g_BAOAB_samples_0_5)//10
    
    N = len(g_BAOAB_samples_0_5)- front

    auto_corr_vector_g_BAOAB_0_5 = []
    
    for i in g_BAOAB_samples_0_5:
        
        auto_corr_vector_g_BAOAB_0_5.append(f(i))
    
    auto_g_BAOAB_0_5 = sm.tsa.acf(auto_corr_vector_g_BAOAB_0_5[front:],nlags = N//50)
    
    IAC_g_BAOAB_0_5 = np.sum(auto_g_BAOAB_0_5)
    
    print('IAC_g_BAOAB=',2*IAC_g_BAOAB_0_5-1)
    
    ESS_percent_g_BAOAB_0_5 = 1/(2*IAC_g_BAOAB_0_5 - 1)
    
    Gradient_p_ESS_g_BAOAB_0_5.append(1/ESS_percent_g_BAOAB_0_5)
    
    
    #Gradient Evaluations per ESS - g-BAOAB - γ = 10.0
    front = len(g_BAOAB_samples_10)//10
    
    N = len(g_BAOAB_samples_10)- front

    auto_corr_vector_g_BAOAB_10 = []
    
    for i in g_BAOAB_samples_10:
        
        auto_corr_vector_g_BAOAB_10.append(f(i))
    
    auto_g_BAOAB_10 = sm.tsa.acf(auto_corr_vector_g_BAOAB_10[front:],nlags = N//50)
    
    IAC_g_BAOAB_10 = np.sum(auto_g_BAOAB_10)
    
    print('IAC_g_BAOAB=',2*IAC_g_BAOAB_10-1)
    
    ESS_percent_g_BAOAB_10 = 1/(2*IAC_g_BAOAB_10 - 1)
    
    Gradient_p_ESS_g_BAOAB_10.append(1/ESS_percent_g_BAOAB_10)


    

In [ ]:
#plt.plot(dt_list,Gradient_p_ESS_DT,'x-',label = '20 step RMHMC')
Num = 100
plt.plot(dt_list[:Num],Gradient_p_ESS_RT_1[:Num], 'x-',label = '1 step RT-RMHMC')
plt.plot(dt_list[:Num],Gradient_p_ESS_RT_2[:Num], 'x-',label = '2 step RT-RMHMC')
plt.plot(dt_list[:Num],Gradient_p_ESS_RT_5[:Num], 'x-',label = '5 step RT-RMHMC')
plt.plot(dt_list[:Num],Gradient_p_ESS_RT_10[:Num], 'x-',label = '10 step RT-RMHMC')
plt.ylabel('Gradient Evaluations per ESS')
plt.xlabel('Step Size')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()
plt.plot(dt_list[:Num],Gradient_p_ESS_g_BAOAB_0_5[:Num],'x-', label = 'g-BAOAB (1 RATTLE) $\gamma = 0.5$')
plt.plot(dt_list[:Num],Gradient_p_ESS_g_BAOAB_2[:Num],'x-', label = 'g-BAOAB (1 RATTLE) $\gamma = 2$')
plt.plot(dt_list[:Num],Gradient_p_ESS_g_BAOAB_10[:Num],'x-', label = 'g-BAOAB (1 RATTLE) $\gamma = 10$')
plt.ylabel('Gradient Evaluations per ESS')
plt.xlabel('Step Size')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()

plt.plot(dt_list[:Num],Gradient_p_ESS_RT_1[:Num], 'x-',label = '1 step RT-RMHMC')
plt.plot(dt_list[:Num],Gradient_p_ESS_RT_2[:Num], 'x-',label = '2 step RT-RMHMC')
plt.plot(dt_list[:Num],Gradient_p_ESS_g_BAOAB_2[:Num],'x-', label = 'g-BAOAB (1 RATTLE) $\gamma = 2$')
plt.plot(dt_list[:Num],Gradient_p_ESS_RT_10[:Num], 'x-',label = '10 step RT-RMHMC')




plt.ylabel('Gradient Evaluations per ESS')
plt.xlabel('Step Size')
plt.yscale('log')
plt.ylabel('Gradient Evaluations per ESS')
plt.xlabel('Step Size')
plt.yscale('log')
plt.legend()
plt.grid()
plt.show()

In [ ]:
Num = 100000
dt_list = [0.1,0.05,0.01]

uniform_list = np.random.uniform(0,1,size=Num)

dt_traj_RT = np.zeros((Num+1,3))

x_init = matrix_to_vec(np.eye(d,p))

counter = 0
for j in dt_list:
    
    
    mean_list_RT_10 = []
    mean_list_DT_10 = []
    mean_list_g_BAOAB_10 = []
    
    print('dt=',j)

    T = 10*j

    #ten_DT_samples, accept_DT = RHMC(Num,j,T,x_init,uniform_list,MH= True)
    ten_RT_samples, gradient_evaluations = RRHMC(Num,j,T,x_init,uniform_list)

    print('RT Acceptance Rate =', accept_RT)
    #print('DT Acceptance Rate =', accept_DT)

    #ten_DT_means = []
    ten_RT_means = []
 



#     mean_curr = 0.
#     for i in ten_DT_samples:

#         mean_curr += f(i)
#         mean_current = mean_curr/(counter2 + 1)
#         ten_DT_means.append(mean_current)
#         counter2 += 1
#     mean_estimate_DT_10 = ten_DT_means[-1]

#     print('DT Mean Estimate =', mean_estimate_DT_10)
    
    counter2 = 0
    mean_curr = 0.
    for i in ten_RT_samples:

        
        mean_curr += f(i)
        mean_current = mean_curr/(counter2 + 1)
        ten_RT_means.append(mean_current)
        dt_traj_RT[counter2,counter] = mean_current
        counter2 += 1
    mean_estimate_RT_10 = ten_RT_means[-1]

    print('RT Mean Estimate =', mean_estimate_RT_10)


#     ######################### g_BAOAB #######################

 
    
    #plt.plot(ten_DT_means,label = '10 step DT-mean')
    plt.plot(ten_RT_means[1000:],label = '10 step RT-mean')

    plt.legend()
    plt.show()
    
    
    counter+=1